In [26]:
import ROOT as r
import math

In [27]:
#MyElectron.h
class MyElectron(r.TLorentzVector):
    def __init__(self, px=0.0, py=0.0, pz=0.0, e=0.0):
        super().__init__(px, py, pz, e)
        self._isolation = 0.0
        self._charge = 0

    def SetIsolation(self, x: float):
        self._isolation = x

    def SetCharge(self, q: int):
        self._charge = q

    def GetIsolation(self) -> float:
        return self._isolation

    def IsIsolated(self) -> bool:
        return self._isolation < 1.0

    def GetCharge(self) -> int:
        return self._charge


In [28]:
#MyJet.h

class MyJet(r.TLorentzVector):
    def __init__(self, px=0.0, py=0.0, pz=0.0, e=0.0):
        super().__init__(px, py, pz, e)
        self._btag = 0.0
        self._jetid = False

    def SetBTagDiscriminator(self, x: float):
        self._btag = x

    def GetBTagDiscriminator(self) -> float:
        return self._btag

    def IsBTagged(self, th: float = 1.74) -> bool:
        return self._btag > th

    def SetJetID(self, jetid: bool):
        self._jetid = jetid

    def GetJetID(self) -> bool:
        return self._jetid

In [29]:
#MyMuon.h

class MyMuon(r.TLorentzVector):
    def __init__(self, px=0.0, py=0.0, pz=0.0, e=0.0):
        super().__init__(px, py, pz, e)
        self._isolation = 0.0
        self._charge = 0

    def SetIsolation(self, x: float):
        self._isolation = x

    def SetCharge(self, q: int):
        self._charge = q

    def GetIsolation(self) -> float:
        return self._isolation

    def IsIsolated(self, relcut: float = 0.05) -> bool:
        pt = self.Pt()
        if pt > 0:
            return (self._isolation / pt) < relcut
        else:
            return False

    def GetCharge(self) -> int:
        return self._charge

In [30]:
#MyPhoton.h

class MyPhoton(r.TLorentzVector):
    def __init__(self, px=0.0, py=0.0, pz=0.0, e=0.0):
        super().__init__(px, py, pz, e)
        self._isolation = 0.0

    def SetIsolation(self, x: float):
        self._isolation = x

    def GetIsolation(self) -> float:
        return self._isolation

    def IsIsolated(self) -> bool:
        return self._isolation < 1.0

In [ ]:
#MyPlotter.h

class Plotter:
    def __init__(self):
        self.data = []
        self.bg = []
        self.signal = []

        self.data_names = []
        self.bg_names = []
        self.signal_names = []

        self.N_histos = 0

    def SetData(self, v, name: str):
        """v: list of TH1F"""
        self.data.append(v)
        self.data_names.append(name)
        self.N_histos = len(v)

    def ClearData(self):
        self.data.clear()
        self.data_names.clear()

    def AddBg(self, v, name: str):
        """v: list of TH1F"""
        self.bg.append(v)
        self.bg_names.append(name)
        self.N_histos = len(v)

    def ClearBg(self):
        self.bg.clear()
        self.bg_names.clear()

    def AddSig(self, v, name: str):
        """v: list of TH1F"""
        self.signal.append(v)
        self.signal_names.append(name)
        self.N_histos = len(v)

    def ClearSig(self):
        self.signal.clear()
        self.signal_names.clear()

    def Plot(self, filename: str = "result.pdf"):
        c = r.TCanvas("c", "c", 800, 600)
        stack = r.THStack("stack", "")

        for i, bg_list in enumerate(self.bg):
            for hist in bg_list:
                stack.Add(hist)

        stack.Draw("HIST")

        for sig_list in self.signal:
            for hist in sig_list:
                hist.SetLineWidth(2)
                hist.SetLineColor(r.kRed)
                hist.Draw("HIST SAME")

        for data_list in self.data:
            for hist in data_list:
                hist.SetMarkerStyle(20)
                hist.SetMarkerSize(1)
                hist.SetLineColor(r.kBlack)
                hist.Draw("E SAME")

        #legend
        legend = r.TLegend(0.7, 0.7, 0.9, 0.9)
        for i, name in enumerate(self.bg_names):
            legend.AddEntry(self.bg[i][0], name, "f")
        for i, name in enumerate(self.signal_names):
            legend.AddEntry(self.signal[i][0], name, "l")
        for i, name in enumerate(self.data_names):
            legend.AddEntry(self.data[i][0], name, "lep")
        legend.Draw()

        c.SaveAs(filename)


In [ ]:
#plotter

from ROOT import (
    TH1F, THStack, TCanvas, TLegend, TStyle, gROOT, kWhite, kRed, kOrange,
    kYellow, kGreen, kCyan, kBlue, kMagenta, kGray, kBlack
)

class Plotter:
    def __init__(self):
        self.data = []          # list of list of TH1F
        self.bg = []            # list of list of TH1F (background)
        self.signal = []        # list of list of TH1F (signal)
        self.data_names = []
        self.bg_names = []
        self.signal_names = []
        self.N_histos = 0

    def __del__(self):
        pass

    def SetData(self, v, name):
        self.data.append(v)
        self.data_names.append(name)
        self.N_histos = len(v)

    def ClearData(self):
        self.data.clear()
        self.data_names.clear()

    def AddBg(self, v, name):
        self.bg.append(v)
        self.bg_names.append(name)
        self.N_histos = len(v)

    def ClearBg(self):
        self.bg.clear()
        self.bg_names.clear()

    def AddSig(self, v, name):
        self.signal.append(v)
        self.signal_names.append(name)
        self.N_histos = len(v)

    def ClearSig(self):
        self.signal.clear()
        self.signal_names.clear()

    def Plot(self, filename="result.pdf"):
        gROOT.Reset()

        MyStyle = TStyle("MyStyle", "My Root Styles")
        MyStyle.SetStatColor(0)
        MyStyle.SetCanvasColor(0)
        MyStyle.SetPadColor(0)
        MyStyle.SetPadBorderMode(0)
        MyStyle.SetCanvasBorderMode(0)
        MyStyle.SetFrameBorderMode(0)
        MyStyle.SetOptStat(0)
        MyStyle.SetStatBorderSize(2)
        MyStyle.SetOptTitle(0)
        MyStyle.SetPadTickX(1)
        MyStyle.SetPadTickY(1)
        MyStyle.SetPadBorderSize(2)
        MyStyle.SetPalette(51, 0)
        MyStyle.SetPadBottomMargin(0.15)
        MyStyle.SetPadTopMargin(0.05)
        MyStyle.SetPadLeftMargin(0.15)
        MyStyle.SetPadRightMargin(0.25)
        MyStyle.SetTitleColor(1)
        MyStyle.SetTitleFillColor(0)
        MyStyle.SetTitleFontSize(0.05)
        MyStyle.SetTitleBorderSize(0)
        MyStyle.SetLineWidth(1)
        MyStyle.SetHistLineWidth(3)
        MyStyle.SetLegendBorderSize(0)
        MyStyle.SetNdivisions(502, "x")
        MyStyle.SetMarkerSize(0.8)
        MyStyle.SetTickLength(0.03)
        MyStyle.SetTitleOffset(1.5, "x")
        MyStyle.SetTitleOffset(1.5, "y")
        MyStyle.SetTitleOffset(1.0, "z")
        MyStyle.SetLabelSize(0.05, "x")
        MyStyle.SetLabelSize(0.05, "y")
        MyStyle.SetLabelSize(0.05, "z")
        MyStyle.SetLabelOffset(0.03, "x")
        MyStyle.SetLabelOffset(0.03, "y")
        MyStyle.SetLabelOffset(0.03, "z")
        MyStyle.SetTitleSize(0.05, "x")
        MyStyle.SetTitleSize(0.05, "y")
        MyStyle.SetTitleSize(0.05, "z")

        gROOT.SetStyle("MyStyle")

        DrawLog = True

        for i in range(self.N_histos):
            hs = None
            Nset = len(self.data) + len(self.bg) + len(self.signal)
            if Nset > 20:
                Nset = 20

            #legend
            l = TLegend(0.76, 0.95 - 0.8 * Nset / 20.0, 1.0, 0.95)
            l.SetFillStyle(1001)
            l.SetFillColor(kWhite)
            l.SetLineColor(kWhite)
            l.SetLineWidth(2)

            if len(self.bg) > 0:
                hs = THStack("hs", self.bg[0][i].GetName())
                for j, bg_set in enumerate(self.bg):
                    h = bg_set[i]
                    # match color scheme
                    color_map = [
                        kRed, kOrange, kYellow, kGreen, kCyan,
                        kBlue, kMagenta, kGray, kGray + 2
                    ]
                    h.SetFillColor(color_map[j] if j < len(color_map) else kBlack)
                    hs.Add(h)
                    l.AddEntry(h, self.bg_names[j], "f")

            c = TCanvas(f"c{i}", f"Canvas {i}", 800, 600)
            c.SetLogy(DrawLog)

            plotname = ""

            if len(self.data) > 0:
                plotname = self.data[0][i].GetName()
                hdata = self.data[0][i]
                hdata.SetMaximum(5 * hdata.GetMaximum())
                hdata.GetXaxis().SetTitleOffset(1.3)
                hdata.GetYaxis().SetTitleOffset(1.3)
                hdata.GetYaxis().SetTitle("Events")
                hdata.GetXaxis().SetNdivisions(505)
                hdata.Draw("")
                l.AddEntry(hdata, self.data_names[0], "p")

                if len(self.bg) > 0:
                    hs.Draw("histsame")

                hdata.SetMarkerStyle(20)
                hdata.Draw("psame")
                l.Draw("same")

            elif len(self.data) == 0 and len(self.bg) > 0:
                plotname = self.bg[0][i].GetName()
                hs.Draw("hist")
                hs.GetXaxis().SetTitleOffset(1.3)
                hs.GetXaxis().SetNdivisions(505)
                hs.GetYaxis().SetTitleOffset(1.3)
                hs.GetYaxis().SetTitle("Events")
                hs.GetXaxis().SetTitle(self.bg[0][i].GetXaxis().GetTitle())
                l.Draw("same")

            if i == 0 and self.N_histos > 1:
                c.Print(f"{filename}(")
            elif i > 0 and i == self.N_histos - 1:
                c.Print(f"{filename})")
            else:
                c.Print(filename)

        print(f"Plots saved to {filename}")


In [33]:
#MyAnalysis.h

class MyAnalysis:
    def __init__(self, SF_b=1.0, weight_factor=1.0, tree=None):
        self.fChain = tree
        self.SF_b = SF_b
        self.weight_factor = weight_factor

        # Event counters
        self.TotalEvents = 0
        self.GeneratedEvents = 0
        self.SelectedEvents = 0
        self.SelectedEvents_triggered = 0

        # Particle collections
        self.Jets = []
        self.Muons = []
        self.Electrons = []
        self.Photons = []

        # TLorentzVectors
        self.hadB = r.TLorentzVector()
        self.lepB = r.TLorentzVector()
        self.hadWq = r.TLorentzVector()
        self.hadWqb = r.TLorentzVector()
        self.lepWl = r.TLorentzVector()
        self.lepWn = r.TLorentzVector()
        self.met = r.TLorentzVector()

        # Branch variables
        self.NJet = 0
        self.Jet_Px = [0.0]*10
        self.Jet_Py = [0.0]*10
        self.Jet_Pz = [0.0]*10
        self.Jet_E = [0.0]*10
        self.Jet_btag = [0.0]*10
        self.Jet_ID = [0.0]*10

        self.NMuon = 0
        self.Muon_Px = [0.0]*5
        self.Muon_Py = [0.0]*5
        self.Muon_Pz = [0.0]*5
        self.Muon_E = [0.0]*5
        self.Muon_Charge = [0]*5
        self.Muon_Iso = [0.0]*5

        self.NElectron = 0
        self.Electron_Px = [0.0]*5
        self.Electron_Py = [0.0]*5
        self.Electron_Pz = [0.0]*5
        self.Electron_E = [0.0]*5
        self.Electron_Charge = [0]*5
        self.Electron_Iso = [0.0]*5

        self.NPhoton = 0
        self.Photon_Px = [0.0]*5
        self.Photon_Py = [0.0]*5
        self.Photon_Pz = [0.0]*5
        self.Photon_E = [0.0]*5
        self.Photon_Iso = [0.0]*5

        self.MET_px = 0.0
        self.MET_py = 0.0

        self.NPrimaryVertices = 0
        self.triggerIsoMu24 = False
        self.EventWeight = 1.0

        # Histograms
        self.h_Mmumu = r.TH1F("h_Mmumu", "Dimuon Mass", 100, 0, 200)
        self.h_Mbqqb_mc = r.TH1F("h_Mbqqb_mc", "bqqb MC Mass", 100, 0, 500)
        self.h_Mbln_mc = r.TH1F("h_Mbln_mc", "bln MC Mass", 100, 0, 500)
        self.h_Mbqqb_reco = r.TH1F("h_Mbqqb_reco", "bqqb Reco Mass", 100, 0, 500)
        self.h_Mbln_reco = r.TH1F("h_Mbln_reco", "bln Reco Mass", 100, 0, 500)

        self.h_NJet = r.TH1F("h_NJet", "Number of Jets", 10, 0, 10)
        self.h_NBJet = r.TH1F("h_NBJet", "Number of b-Jets", 10, 0, 10)
        self.h_NMuon = r.TH1F("h_NMuon", "Number of Muons", 5, 0, 5)
        self.h_NElectron = r.TH1F("h_NElectron", "Number of Electrons", 5, 0, 5)

        self.h_Jet1_Pt = r.TH1F("h_Jet1_Pt", "Jet1 Pt", 100, 0, 500)
        self.h_Jet1_Eta = r.TH1F("h_Jet1_Eta", "Jet1 Eta", 50, -5, 5)
        self.h_Jet2_Pt = r.TH1F("h_Jet2_Pt", "Jet2 Pt", 100, 0, 500)
        self.h_Jet2_Eta = r.TH1F("h_Jet2_Eta", "Jet2 Eta", 50, -5, 5)
        self.h_Jet3_Pt = r.TH1F("h_Jet3_Pt", "Jet3 Pt", 100, 0, 500)
        self.h_Jet3_Eta = r.TH1F("h_Jet3_Eta", "Jet3 Eta", 50, -5, 5)

        self.h_BJet1_Pt = r.TH1F("h_BJet1_Pt", "BJet1 Pt", 100, 0, 500)
        self.h_BJet1_Eta = r.TH1F("h_BJet1_Eta", "BJet1 Eta", 50, -5, 5)
        self.h_BJet2_Pt = r.TH1F("h_BJet2_Pt", "BJet2 Pt", 100, 0, 500)
        self.h_BJet2_Eta = r.TH1F("h_BJet2_Eta", "BJet2 Eta", 50, -5, 5)

        self.h_Muon1_Pt = r.TH1F("h_Muon1_Pt", "Muon1 Pt", 100, 0, 200)
        self.h_Muon1_Eta = r.TH1F("h_Muon1_Eta", "Muon1 Eta", 50, -5, 5)
        self.h_Muon1_Iso = r.TH1F("h_Muon1_Iso", "Muon1 Iso", 50, 0, 1)
        self.h_Muon2_Pt = r.TH1F("h_Muon2_Pt", "Muon2 Pt", 100, 0, 200)
        self.h_Muon2_Eta = r.TH1F("h_Muon2_Eta", "Muon2 Eta", 50, -5, 5)
        self.h_Muon2_Iso = r.TH1F("h_Muon2_Iso", "Muon2 Iso", 50, 0, 1)

        self.h_Electron1_Pt = r.TH1F("h_Electron1_Pt", "Electron1 Pt", 100, 0, 200)
        self.h_Electron1_Eta = r.TH1F("h_Electron1_Eta", "Electron1 Eta", 50, -5, 5)

        self.h_MET = r.TH1F("h_MET", "MET", 100, 0, 500)
        self.h_minDeltaPhi_MET_Muon = r.TH1F("h_minDeltaPhi_MET_Muon", "minDeltaPhi MET-Muon", 50, 0, 3.2)
        self.h_minDeltaPhi_MET_BJet = r.TH1F("h_minDeltaPhi_MET_BJet", "minDeltaPhi MET-BJet", 50, 0, 3.2)
        self.h_nPV = r.TH1F("h_nPV", "Number of PV", 50, 0, 50)

        self.h_selectedEvents_Muon1_Pt = r.TH1F("h_selectedEvents_Muon1_Pt", "Muon1 Pt (selected)", 100, 0, 200)
        self.h_selectedEvents_triggered_Muon1_Pt = r.TH1F("h_selectedEvents_triggered_Muon1_Pt", "Muon1 Pt (triggered)", 100, 0, 200)

        self.histograms = [
            self.h_NJet, self.h_NBJet, self.h_NMuon, self.h_NElectron,
            self.h_Jet1_Pt, self.h_Jet1_Eta, self.h_Jet2_Pt, self.h_Jet2_Eta, self.h_Jet3_Pt, self.h_Jet3_Eta,
            self.h_BJet1_Pt, self.h_BJet1_Eta, self.h_BJet2_Pt, self.h_BJet2_Eta,
            self.h_Muon1_Pt, self.h_Muon1_Eta, self.h_Muon1_Iso,
            self.h_Muon2_Pt, self.h_Muon2_Eta, self.h_Muon2_Iso,
            self.h_Electron1_Pt, self.h_Electron1_Eta,
            self.h_MET, self.h_minDeltaPhi_MET_Muon, self.h_minDeltaPhi_MET_BJet, self.h_nPV,
            self.h_selectedEvents_Muon1_Pt, self.h_selectedEvents_triggered_Muon1_Pt
        ]

    def BuildEvent(self, entry):
        """Fill particle collections for one entry."""
        self.fChain.GetEntry(entry)

        # Jets
        self.Jets = []
        for i in range(self.NJet):
            jet = MyJet(self.Jet_Px[i], self.Jet_Py[i], self.Jet_Pz[i], self.Jet_E[i])
            jet.SetBTagDiscriminator(self.Jet_btag[i])
            jet.SetJetID(bool(self.Jet_ID[i]))
            self.Jets.append(jet)

        # Muons
        self.Muons = []
        for i in range(self.NMuon):
            mu = MyMuon(self.Muon_Px[i], self.Muon_Py[i], self.Muon_Pz[i], self.Muon_E[i])
            mu.SetCharge(self.Muon_Charge[i])
            mu.SetIsolation(self.Muon_Iso[i])
            self.Muons.append(mu)

        # Electrons
        self.Electrons = []
        for i in range(self.NElectron):
            el = MyElectron(self.Electron_Px[i], self.Electron_Py[i], self.Electron_Pz[i], self.Electron_E[i])
            el.SetCharge(self.Electron_Charge[i])
            el.SetIsolation


In [34]:
def main():
    lumi = 50.0

    # Load and process data
    A = MyAnalysis()
    ch = r.TChain("events")
    ch.Add("files_for_python_for_python_for_python_for_python_for_python_for_python_for_python_for_python_for_python_for_python_for_python_for_python_for_python_for_python_for_python_for_python_for_python_for_python/data.root")
    ch.Process(A)

    B = MyAnalysis()
    ch2 = r.TChain("events")
    ch2.Add("files_for_python/ttbar.root")
    ch2.Process(B)

    C = MyAnalysis()
    ch3 = r.TChain("events")
    ch3.Add("files_for_python/wjets.root")
    ch3.Process(C)

    D = MyAnalysis()
    ch4 = r.TChain("events")
    ch4.Add("files_for_python/dy.root")
    ch4.Process(D)

    E = MyAnalysis()
    ch5 = r.TChain("events")
    ch5.Add("files_for_python/ww.root")
    ch5.Process(E)

    F = MyAnalysis()
    ch6 = r.TChain("events")
    ch6.Add("files_for_python/wz.root")
    ch6.Process(F)

    G = MyAnalysis()
    ch7 = r.TChain("events")
    ch7.Add("files_for_python/zz.root")
    ch7.Process(G)

    H = MyAnalysis()
    ch8 = r.TChain("events")
    ch8.Add("files_for_python/qcd.root")
    ch8.Process(H)

    I = MyAnalysis()
    ch9 = r.TChain("events")
    ch9.Add("files_for_python/single_top.root")
    ch9.Process(I)

    # Plot data and backgrounds
    P = Plotter()
    P.SetData(A.histograms, "Data")
    P.AddBg(B.histograms, "TTbar")
    P.AddBg(C.histograms, "Wjets")
    P.AddBg(D.histograms, "DY")
    P.AddBg(E.histograms, "WW")
    P.AddBg(F.histograms, "WZ")
    P.AddBg(G.histograms, "ZZ")
    P.AddBg(H.histograms, "QCD")
    P.AddBg(I.histograms, "single Top")
    P.Plot("results.pdf")

    # Plot MC histograms
    P_MC = Plotter()
    P_MC.AddBg(B.histograms_MC, "TTbar")
    P_MC.AddBg(C.histograms_MC, "Wjets")
    P_MC.AddBg(D.histograms_MC, "DY")
    P_MC.AddBg(E.histograms_MC, "WW")
    P_MC.AddBg(F.histograms_MC, "WZ")
    P_MC.AddBg(G.histograms_MC, "ZZ")
    P_MC.AddBg(H.histograms_MC, "QCD")
    P_MC.AddBg(I.histograms_MC, "single Top")
    P_MC.Plot("results_MC.pdf")

    # Trigger efficiency plot
    c2 = r.TCanvas("c2", "c2", 600, 600)
    h_trigg_eff = r.TGraphAsymmErrors(B.h_selectedEvents_triggered_Muon1_Pt,
                                      B.h_selectedEvents_Muon1_Pt)
    h_trigg_eff.GetXaxis().SetTitle("Muon p_{T} [GeV]")
    h_trigg_eff.GetYaxis().SetTitle("Trigger efficiency")
    h_trigg_eff.UseCurrentStyle()
    h_trigg_eff.Draw("AP")
    c2.Print("Trigger_eff.pdf")

    # Cross section calculation
    print("Monte Carlo:")
    print(f"Signal Number of generated events: {B.GeneratedEvents}")
    print(f"Signal Number of selected events: {B.SelectedEvents}")
    print(f"Signal Number of selected and triggered events: {B.SelectedEvents_triggered}\n")

    acc = B.SelectedEvents / B.GeneratedEvents
    print(f"Signal Acceptance: {acc}")
    trigg_eff = B.SelectedEvents_triggered / B.SelectedEvents
    print(f"Trigger efficiency: {trigg_eff}\n")

    N_bg = (C.SelectedEvents_triggered + D.SelectedEvents_triggered +
            E.SelectedEvents_triggered + F.SelectedEvents_triggered +
            G.SelectedEvents_triggered + H.SelectedEvents_triggered +
            I.SelectedEvents_triggered)
    print(f"Background Number of selected and triggered events: {N_bg}")
    purity = B.SelectedEvents_triggered / (B.SelectedEvents_triggered + N_bg)
    print(f"Purity: {purity}\n")

    print("Data:")
    print(f"Number of selected and triggered events: {A.SelectedEvents_triggered}")
    N_bgSub = A.SelectedEvents_triggered - A.SelectedEvents_triggered * (1.0 - purity)
    print(f"Number of selected and triggered events (bg subtracted): {N_bgSub}")
    N_corr = N_bgSub / acc / trigg_eff
    print(f"Acceptance and trigg efficiency corrected yield: {N_corr}")
    print(f"Cross section [pb]: {N_corr / lumi}")
    print("-----------------------------------------------")



In [ ]:
class MyAnalysis:
    def __init__(self):
        # Event containers
        self.Muons = []
        self.Electrons = []
        self.Photons = []
        self.Jets = []

        # MC particles
        self.hadB = r.TLorentzVector()
        self.lepB = r.TLorentzVector()
        self.hadWq = r.TLorentzVector()
        self.hadWqb = r.TLorentzVector()
        self.lepWl = r.TLorentzVector()
        self.lepWn = r.TLorentzVector()
        self.met = r.TLorentzVector()

        # Event weight
        self.EventWeight = 1.0

        # Histograms
        self.histograms = []
        self.histograms_MC = []

    def BuildEvent(self):
        self.Muons.clear()
        for i in range(self.NMuon):
            muon = MyMuon(self.Muon_Px[i], self.Muon_Py[i], self.Muon_Pz[i], self.Muon_E[i])
            muon.SetIsolation(self.Muon_Iso[i])
            muon.SetCharge(self.Muon_Charge[i])
            self.Muons.append(muon)

        self.Electrons.clear()
        for i in range(self.NElectron):
            electron = MyElectron(self.Electron_Px[i], self.Electron_Py[i],
                                  self.Electron_Pz[i], self.Electron_E[i])
            electron.SetIsolation(self.Electron_Iso[i])
            electron.SetCharge(self.Electron_Charge[i])
            self.Electrons.append(electron)

        self.Photons.clear()
        for i in range(self.NPhoton):
            photon = MyPhoton(self.Photon_Px[i], self.Photon_Py[i],
                              self.Photon_Pz[i], self.Photon_E[i])
            photon.SetIsolation(self.Photon_Iso[i])
            self.Photons.append(photon)

        self.Jets.clear()
        for i in range(self.NJet):
            jet = MyJet(self.Jet_Px[i], self.Jet_Py[i], self.Jet_Pz[i], self.Jet_E[i])
            jet.SetBTagDiscriminator(self.Jet_btag[i])
            jet.SetJetID(self.Jet_ID[i])
            self.Jets.append(jet)

        self.hadB.SetXYZM(self.MChadronicBottom_px, self.MChadronicBottom_py, self.MChadronicBottom_pz, 4.8)
        self.lepB.SetXYZM(self.MCleptonicBottom_px, self.MCleptonicBottom_py, self.MCleptonicBottom_pz, 4.8)
        self.hadWq.SetXYZM(self.MChadronicWDecayQuark_px, self.MChadronicWDecayQuark_py, self.MChadronicWDecayQuark_pz, 0.0)
        self.hadWqb.SetXYZM(self.MChadronicWDecayQuarkBar_px, self.MChadronicWDecayQuarkBar_py, self.MChadronicWDecayQuarkBar_pz, 0.0)
        self.lepWl.SetXYZM(self.MClepton_px, self.MClepton_py, self.MClepton_pz, 0.0)
        self.lepWn.SetXYZM(self.MCneutrino_px, self.MCneutrino_py, self.MCneutrino_pz, 0.0)
        self.met.SetXYZM(self.MET_px, self.MET_py, 0., 0.)

        self.EventWeight *= self.weight_factor

    def SlaveBegin(self):
        # example
        self.h_Mmumu = r.TH1F("Mmumu", "Invariant di-muon mass", 60, 60, 120)
        self.h_Mmumu.GetXaxis().SetTitle("m_{#mu#mu}")
        self.h_Mmumu.Sumw2()
        self.histograms.append(self.h_Mmumu)
        self.histograms_MC.append(self.h_Mmumu)


In [ ]:
class MyAnalysis:

    def Process(self, entry):
        self.TotalEvents += 1
        self.GetEntry(entry)

        if self.TotalEvents % 10000 == 0:
            print(f"Next event -----> {self.TotalEvents}")

        self.BuildEvent()

        # Muon cuts
        MuonPtCut = 25.0
        MuonRelIsoCut = 0.10

        # Count isolated muons
        N_IsoMuon = 0
        muon1 = None
        muon2 = None
        for mu in self.Muons:
            if mu.IsIsolated(MuonRelIsoCut):
                N_IsoMuon += 1
                if N_IsoMuon == 1: muon1 = mu
                if N_IsoMuon == 2: muon2 = mu

        self.h_NMuon.Fill(N_IsoMuon, self.EventWeight)

        if N_IsoMuon > 1 and self.triggerIsoMu24:
            if muon1.Pt() > MuonPtCut:
                self.h_Mmumu.Fill((muon1 + muon2).M(), self.EventWeight)

        # Count b-jets
        N_BJet = sum(1 for j in self.Jets if j.IsBTagged())

        # Jets selection for selected isolated muon
        N_Jet = 0
        N_BJet_tmp = 0
        if self.triggerIsoMu24 and N_IsoMuon > 0:
            if muon1.Pt() > MuonPtCut:
                self.h_nPV.Fill(self.NPrimaryVertices, self.EventWeight)

                for j in self.Jets:
                    if not j.GetJetID(): 
                        continue
                    N_Jet += 1
                    # Fill leading jets
                    if N_Jet == 1:
                        self.h_Jet1_Pt.Fill(j.Pt(), self.EventWeight)
                        self.h_Jet1_Eta.Fill(j.Eta(), self.EventWeight)
                    elif N_Jet == 2:
                        self.h_Jet2_Pt.Fill(j.Pt(), self.EventWeight)
                        self.h_Jet2_Eta.Fill(j.Eta(), self.EventWeight)
                    elif N_Jet == 3:
                        self.h_Jet3_Pt.Fill(j.Pt(), self.EventWeight)
                        self.h_Jet3_Eta.Fill(j.Eta(), self.EventWeight)

                    if j.IsBTagged():
                        N_BJet_tmp += 1
                        if N_BJet_tmp == 1:
                            self.h_BJet1_Pt.Fill(j.Pt(), self.EventWeight * self.SF_b)
                            self.h_BJet1_Eta.Fill(j.Eta(), self.EventWeight * self.SF_b)
                        elif N_BJet_tmp == 2:
                            self.h_BJet2_Pt.Fill(j.Pt(), self.EventWeight * self.SF_b**2)
                            self.h_BJet2_Eta.Fill(j.Eta(), self.EventWeight * self.SF_b**2)

                self.h_NBJet.Fill(N_BJet, self.EventWeight * self.SF_b**N_BJet)
                self.h_NJet.Fill(N_Jet, self.EventWeight)

                # Isolated muons histograms
                N_IsoMuon_counter = 0
                for mu in self.Muons:
                    if mu.IsIsolated(MuonRelIsoCut):
                        N_IsoMuon_counter += 1
                        if N_IsoMuon_counter == 1:
                            self.h_Muon1_Pt.Fill(mu.Pt(), self.EventWeight)
                            self.h_Muon1_Eta.Fill(mu.Eta(), self.EventWeight)
                            self.h_Muon1_Iso.Fill(mu.GetIsolation(), self.EventWeight)
                        elif N_IsoMuon_counter == 2:
                            self.h_Muon2_Pt.Fill(mu.Pt(), self.EventWeight)
                            self.h_Muon2_Eta.Fill(mu.Eta(), self.EventWeight)
                            self.h_Muon2_Iso.Fill(mu.GetIsolation(), self.EventWeight)

                # Isolated electrons
                N_IsoElectron_counter = 0
                for ele in self.Electrons:
                    if ele.GetIsolation() / ele.Pt() < MuonRelIsoCut:
                        N_IsoElectron_counter += 1
                        if N_IsoElectron_counter == 1:
                            self.h_Electron1_Pt.Fill(ele.Pt(), self.EventWeight)
                            self.h_Electron1_Eta.Fill(ele.Eta(), self.EventWeight)
                self.h_NElectron.Fill(N_IsoElectron_counter, self.EventWeight)

                # MET
                self.h_MET.Fill(self.met.Pt(), self.EventWeight)

        # Trigger / selected events
        if len(self.Muons) == 1 and self.Muons[0].IsIsolated(MuonRelIsoCut):
            self.h_selectedEvents_Muon1_Pt.Fill(self.Muons[0].Pt(), self.EventWeight)
            if self.triggerIsoMu24:
                self.h_selectedEvents_triggered_Muon1_Pt.Fill(self.Muons[0].Pt(), self.EventWeight)

        # Acceptance efficiency
        self.GeneratedEvents += self.EventWeight
        IsSelected = False
        if N_IsoMuon == 1 and muon1.Pt() > MuonPtCut:
            NBJet = sum(1 for j in self.Jets if j.IsBTagged())
            if NBJet > 1:
                self.SelectedEvents += self.EventWeight * self.SF_b**2
                if self.triggerIsoMu24:
                    self.SelectedEvents_triggered += self.EventWeight * self.SF_b**2
                    IsSelected = True

        # MC top mass
        self.h_Mbqqb_mc.Fill((self.hadB + self.hadWq + self.hadWqb).M(), self.EventWeight)
        self.h_Mbqqb_mc.Fit("gaus")
        self.h_Mbln_mc.Fill((self.lepB + self.lepWl + self.lepWn).M(), self.EventWeight)
        self.h_Mbln_mc.Fit("gaus")

        # Reco top mass (hadronic & leptonic)
        if IsSelected:
            # Hadronic top
            for i, bjet in enumerate(self.Jets):
                if not bjet.IsBTagged():
                    continue
                for j, jet2 in enumerate(self.Jets):
                    if j <= i or jet2.IsBTagged():
                        continue
                    for k, jet3 in enumerate(self.Jets):
                        if k <= j or k == i or jet3.IsBTagged():
                            continue
                        W_mass = (jet2 + jet3).M()
                        if 70 < W_mass < 95:
                            self.h_Mbqqb_reco.Fill((bjet + jet2 + jet3).M(), self.EventWeight)

            # Leptonic top
            for bjet in self.Jets:
                if bjet.IsBTagged():
                    px = self.met.Px()
                    py = self.met.Py()
                    mW = 80.379
                    A = mW**2 / 2 + muon1.Px()*px + muon1.Py()*py
                    F = 2*A*muon1.Pz() / (muon1.Px()**2 + muon1.Py()**2)
                    G = (muon1.E()**2 * self.met.Pt()**2 - A**2) / (muon1.Px()**2 + muon1.Py()**2)
                    D = muon1.E()**2 * (A**2 - self.met.Pt()**2 * (muon1.E()**2 - muon1.Pz()**2))

                    if D >= 0:
                        pz = F + math.sqrt(D)
                        E = math.sqrt(px**2 + py**2 + pz**2)
                        neutrino1 = r.TLorentzVector(px, py, pz, E)
                        self.h_Mbln_reco.Fill((bjet + muon1 + neutrino1).M(), self.EventWeight)

                    if D > 0:
                        pz = F - math.sqrt(D)
                        E = math.sqrt(px**2 + py**2 + pz**2)
                        neutrino2 = r.TLorentzVector(px, py, pz, E)
                        self.h_Mbln_reco.Fill((bjet + muon1 + neutrino2).M(), self.EventWeight)

        return True
